# Databricks Certified Data Engineer Associate — Exam Outline Lab

Companion notebook for the **Databricks Certified Data Engineer Associate – Exam Guide (August 2026)**.
https://www.databricks.com/sites/default/files/2026-10/Databricks-Certified-Data-Engineer-Associate-Exam-Guide-August-2026.pdf 

Every exam objective from the *Exam Outline* is reproduced as a Markdown cell (original wording) followed by a **PySpark / Python cell that runs and demonstrates the topic**.

| Section | Weight |
|---|---|
| 1. Databricks Intelligence Platform | 6% |
| 2. Data Ingestion and Loading | 21% |
| 3. Data Transformation and Modeling | 22% |
| 4. Working with Lakeflow Jobs | 16% |
| 5. Implementing CI/CD | 10% |
| 6. Troubleshooting, Monitoring, and Optimization | 10% |
| 7. Governance and Security | 15% |

**How to use**
1. Attach **serverless** compute (recommended) or a UC-enabled cluster on a recent Databricks Runtime and run the *Setup* cell below once.
2. Everything is created in a sandbox schema (widget `schema`, default `de_associate_demo`) inside your current catalog. Nothing outside it is touched.
3. Cells that would create jobs / pipelines / ABAC policies only **print** their definition unless you set the widget `run_side_effects` to `true`.
4. Cells that need features your workspace may not have (system tables, external locations, Lakeflow Connect connections, SQL-warehouse-only objects) are wrapped so they report what is missing and the notebook keeps going.

> Cells are written to be mostly independent: each one only depends on the *Setup* cell.

In [0]:
# Setup — run this cell first. All objects live in a sandbox schema.
import json, os, re, sys, time, shutil, subprocess, textwrap, io, contextlib
from pyspark.sql import functions as F, Window
from pyspark.sql.types import *

dbutils.widgets.text("catalog", "", "Catalog (blank = current catalog)")
dbutils.widgets.text("schema", "de_associate_demo", "Sandbox schema")
dbutils.widgets.dropdown("run_side_effects", "false", ["false", "true"], "Create jobs/pipelines/policies?")

CATALOG = dbutils.widgets.get("catalog") or spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = dbutils.widgets.get("schema")
RUN_SIDE_EFFECTS = dbutils.widgets.get("run_side_effects") == "true"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")
spark.sql("CREATE VOLUME IF NOT EXISTS landing")
VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/landing"


def conf(key):
    """Read a Spark conf, returning None when it is not available (e.g. on serverless)."""
    try:
        return spark.conf.get(key)
    except Exception:
        return None


def set_conf(key, value):
    try:
        spark.conf.set(key, value)
        return True
    except Exception as e:
        print(f"  cannot set {key}: {type(e).__name__}")
        return False


def try_run(label, fn):
    """Run fn(); print a short message instead of failing the whole notebook."""
    try:
        return fn()
    except Exception as e:
        print(f"[{label}] skipped / failed -> {type(e).__name__}: {str(e)[:300]}")
        return None


def write_json_lines(path, rows):
    dbutils.fs.put(path, "\n".join(json.dumps(r) for r in rows), True)


def get_workspace_client():
    from databricks.sdk import WorkspaceClient
    return WorkspaceClient()


def notebook_path():
    try:
        return dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    except Exception:
        return "/Workspace/Users/<you>/Databricks Data Engineer Associate"


def show_spec(spec):
    print(json.dumps(spec, indent=2))


def create_job_if_allowed(spec):
    if not RUN_SIDE_EFFECTS:
        print("run_side_effects=false -> job definition printed only, nothing created.")
        return None
    resp = get_workspace_client().api_client.do("POST", "/api/2.1/jobs/create", body=spec)
    print("Created job:", resp)
    return resp


print(f"Catalog: {CATALOG} | Schema: {SCHEMA} | Volume: {VOLUME} | run_side_effects={RUN_SIDE_EFFECTS}")

---
## Section 1: Databricks Intelligence Platform (6%)

#### Objective 1.1
> Understand the core components of the Databricks Platform, such as its architecture, Delta Lake, and Unity Catalog.

**Databricks Platform** - Databricks is a unified data and AI platform that runs both analytical and operational workloads on one open, governed foundation. It enables organizations to build BI and AI applications, stream and analyze data, and process daily operations with consistent security, performance, and governance — all without adding complexity or having to copy data. (source: https://www.databricks.com/)

**Delta Lake** - The open-source storage layer that provides the foundation for lakehouse tables: it extends Parquet data files with a file-based transaction log for ACID transactions and scalable metadata handling, and unifies streaming and batch operations on a single copy of data. It is the default format for all tables on Databricks. (source: https://docs.databricks.com/aws/en/delta/)

**Unity Catalog** - Databricks' unified governance layer for data and AI: a single place to control access, discover assets, track lineage, classify sensitive data, and monitor quality across the organization. Its hierarchy is metastore > catalog > schema > table/view/volume/function/model. (source: https://docs.databricks.com/aws/en/data-governance/)

In [0]:
# Architecture: control plane (UI, jobs, notebooks) vs compute plane (your clusters / serverless) over cloud storage
print("Spark version :", spark.version)
print("Runtime       :", os.environ.get("DATABRICKS_RUNTIME_VERSION", "unknown"))
display(spark.sql("""
    SELECT current_metastore() AS metastore, current_catalog() AS catalog,
           current_schema() AS schema, current_user() AS user
"""))

# Delta Lake: open table format = Parquet files + transaction log (ACID, time travel, schema enforcement)
spark.range(5).withColumn("squared", F.col("id") ** 2).write.mode("overwrite").saveAsTable("demo_delta")
spark.sql("INSERT INTO demo_delta VALUES (99, 9801)")
display(spark.sql("DESCRIBE HISTORY demo_delta").select("version", "timestamp", "operation", "operationMetrics"))
display(spark.sql("SELECT count(*) AS rows_at_version_0 FROM demo_delta VERSION AS OF 0"))   # time travel
display(spark.sql("DESCRIBE DETAIL demo_delta").select("format", "location", "numFiles", "sizeInBytes"))

# Unity Catalog: one governance layer -> metastore > catalog > schema > table/view/volume/function/model
display(spark.sql("SHOW CATALOGS"))
display(spark.sql(f"SHOW TABLES IN `{CATALOG}`.`{SCHEMA}`"))
display(spark.sql("DESCRIBE TABLE EXTENDED demo_delta")
        .filter("col_name IN ('Catalog', 'Database', 'Table', 'Type', 'Provider', 'Location')"))

#### Objective 1.2
> Understand Databricks Platform's compute services, including their characteristics, limitations, and cost models, and select the most suitable option for each workload use case.

**Serverless compute** - On-demand, automatically managed compute that scales with workload requirements, with no infrastructure to configure, deploy, or leave idle; you are billed for the compute actually used. (source: https://docs.databricks.com/aws/en/compute/)

**SQL warehouse** - The SQL-tuned compute resource powering Databricks SQL: scalable SQL compute decoupled from storage, serving the SQL editor, dashboards, and BI tools with ANSI SQL on lakehouse data. (source: https://docs.databricks.com/aws/en/sql/)

**Classic compute** - Clusters you fully configure (instance types, libraries, Spark configs, init scripts). All-purpose clusters serve interactive work at the highest DBU rate; jobs clusters are created per job run and terminated afterwards, at a lower DBU rate. (source: https://docs.databricks.com/aws/en/compute/)

In [0]:
# 1) Which compute am I on right now?
runtime = os.environ.get("DATABRICKS_RUNTIME_VERSION", "unknown")
cluster_id = conf("spark.databricks.clusterUsageTags.clusterId")
is_serverless = cluster_id is None or runtime.startswith("client.")
print(f"Runtime: {runtime}")
if is_serverless:
    print("Detected: SERVERLESS compute (no cluster id, Databricks-managed, versionless runtime)")
else:
    print(f"Detected: CLASSIC cluster {cluster_id}")
    print("  node type :", conf("spark.databricks.clusterUsageTags.clusterNodeType"))
    print("  workers   :", conf("spark.databricks.clusterUsageTags.clusterTargetWorkers"))

# 2) Characteristics, limitations, cost model and best fit of each compute option
compute_options = [
    ("Serverless compute (notebooks / jobs / pipelines)",
     "Fully managed by Databricks; starts in seconds; auto-scaling and automatic runtime upgrades; no cluster sizing",
     "Python and SQL only; no RDD API, init scripts or custom containers; only a small allow-list of Spark configs can be set",
     "Billed in DBUs for the compute actually used (infrastructure cost included); no idle cluster to pay for",
     "Ad-hoc / interactive work, short or bursty jobs, teams that want zero cluster ops"),
    ("SQL warehouse (serverless / pro / classic)",
     "SQL-only engine tuned for BI and ELT SQL; Photon; auto-stop; T-shirt sizes; serverless starts in seconds",
     "SQL only (no PySpark / Python); classic and pro warehouses have slower start-up than serverless",
     "DBU-per-hour by warehouse size while running (serverless includes cloud infra; classic/pro add VM cost)",
     "Dashboards, BI tools via JDBC/ODBC, SQL analytics, SQL-based transformations"),
    ("All-purpose compute (classic cluster)",
     "Interactive, shareable, fully configurable: any library, Spark config, init script, instance type, GPU",
     "You size and tune it; slower start (minutes); easy to leave running; highest DBU rate",
     "All-purpose DBU rate + cloud VM cost for as long as the cluster is up (use auto-termination)",
     "Exploration and development needing custom libraries / configs, ML with special hardware"),
    ("Jobs compute (classic job cluster)",
     "Ephemeral cluster created for a job run and terminated afterwards; fully configurable",
     "Start-up latency for every run; not for interactive use",
     "Lower DBU rate than all-purpose + cloud VM cost, only during the run",
     "Scheduled, predictable production pipelines that need custom configuration or libraries"),
]
display(spark.createDataFrame(
    compute_options,
    "compute STRING, characteristics STRING, limitations STRING, cost_model STRING, best_for STRING"))

# 3) A limitation in action: static JVM settings cannot be changed from a notebook
for key, value in [("spark.sql.session.timeZone", conf("spark.sql.session.timeZone") or "UTC"),
                   ("spark.executor.memory", "8g")]:
    try:
        spark.conf.set(key, value)
        print(f"set {key}={value}  -> accepted")
    except Exception as e:
        print(f"set {key}={value}  -> rejected ({type(e).__name__}); configure it on a classic cluster instead")

# 4) Cost model = DBUs x price (+ VM cost on classic). Placeholder numbers - plug in your own contract prices.
def estimate_cost(dbu_per_hour, hours, usd_per_dbu, vm_usd_per_hour=0.0):
    return round(dbu_per_hour * hours * usd_per_dbu + vm_usd_per_hour * hours, 2)

print("ILLUSTRATIVE 2h job, classic job cluster :", estimate_cost(8, 2, 0.15, vm_usd_per_hour=3.0))
print("ILLUSTRATIVE 2h job, serverless          :", estimate_cost(8, 2, 0.35))

---
## Section 2: Data Ingestion and Loading (21%)

#### Objective 2.1
> Enable and detail data ingestion patterns, including batch, streaming, and incremental loading, and import data from sources such as local files, Lakeflow Connect standard connectors, and Lakeflow Connect managed connectors.

**Batch, incremental, and streaming ingestion** - Batch loading processes all files present at run time in one go; incremental loading moves only what is new since the last run (via a checkpoint or high-water mark); streaming ingestion continuously processes data as it arrives. (source: https://docs.databricks.com/aws/en/ingestion/overview/)

**Lakeflow Connect standard connectors** - Ingestion connectors built on open frameworks (for example Auto Loader for cloud storage, Kafka) that you configure and schedule yourself, giving full control and customization. (source: https://docs.databricks.com/aws/en/ingestion/overview/)

**Lakeflow Connect managed connectors** - Fully-managed ingestion pipelines from enterprise applications and databases: a Unity Catalog connection plus an ingestion pipeline, with Databricks managing orchestration, retries, and infrastructure. (source: https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/)

In [0]:
base = f"{VOLUME}/ingest_patterns"
dbutils.fs.rm(base, True)

# --- Batch: read everything that is there, in one go -------------------------------------
write_json_lines(f"{base}/batch/orders_001.json", [{"order_id": i, "amount": i * 10.0} for i in range(1, 6)])
display(spark.read.json(f"{base}/batch"))

# --- Local file -> Volume -> table (small files; for big data use cloud storage / connectors) ---
local_csv = "/tmp/local_customers.csv"
with open(local_csv, "w") as f:
    f.write("customer_id,name\n1,Ada\n2,Grace\n")
os.makedirs(f"{base}/local", exist_ok=True)
shutil.copy(local_csv, f"{base}/local/local_customers.csv")
display(spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/local"))

# --- Streaming + incremental: a stream with availableNow processes only what is new -------
spark.sql("CREATE OR REPLACE TABLE events_src (event_id INT, ts TIMESTAMP)")
spark.sql("INSERT INTO events_src VALUES (1, current_timestamp()), (2, current_timestamp())")
spark.sql("CREATE OR REPLACE TABLE events_tgt (event_id INT, ts TIMESTAMP)")
ckpt = f"{base}/_ckpt_events"

def run_incremental():
    q = (spark.readStream.table("events_src").writeStream
         .option("checkpointLocation", ckpt)
         .trigger(availableNow=True)
         .toTable("events_tgt"))
    q.awaitTermination()
    return spark.table("events_tgt").count()

print("rows in target after run 1:", run_incremental())      # 2
spark.sql("INSERT INTO events_src VALUES (3, current_timestamp())")
print("rows in target after run 2:", run_incremental())      # 3 -> only the new row was processed

# --- Incremental batch with a high-water mark (when you cannot use a stream) ---------------
hwm = spark.table("events_tgt").agg(F.max("ts")).first()[0]
print("next batch would load rows with ts >", hwm)

# Lakeflow Connect: *standard* connectors (cloud storage, Kafka, ... built on Auto Loader / streaming)
# and *managed* connectors (SaaS apps and databases, fully managed ingestion pipelines) are configured
# in the UI / API -> see Objective 2.4.

#### Objective 2.2
> Use the COPY INTO command to incrementally load files from cloud object storage (ADLS/S3/GCS) into Unity Catalog-governed tables.

**COPY INTO** - A retriable, idempotent SQL command that loads data from a file location into a Delta table; files already loaded are skipped on subsequent runs, so it can be safely re-run on a schedule. Supports CSV, JSON, XML, Avro, ORC, Parquet and text files from S3, ADLS, GCS, and Unity Catalog volumes. (source: https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/copy-into/)

In [0]:
src = f"{VOLUME}/copy_into_src"      # in production: abfss://container@acct.dfs.core.windows.net/..., s3://..., gs://...
dbutils.fs.rm(src, True)
write_json_lines(f"{src}/batch1.json", [{"id": 1, "name": "a"}, {"id": 2, "name": "b"}])

spark.sql("DROP TABLE IF EXISTS copy_into_demo")
spark.sql("CREATE TABLE copy_into_demo")          # schemaless Delta table: COPY INTO infers the schema

copy_sql = f"""
COPY INTO copy_into_demo
FROM '{src}'
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
"""
print("Run 1 - loads both rows:")
display(spark.sql(copy_sql))
print("Run 2 - idempotent, already-loaded files are skipped:")
display(spark.sql(copy_sql))

write_json_lines(f"{src}/batch2.json", [{"id": 3, "name": "c", "extra": "new column"}])
print("Run 3 - only the new file is loaded, schema merged:")
display(spark.sql(copy_sql))
display(spark.table("copy_into_demo").orderBy("id"))

#### Objective 2.3
> Use Auto Loader with schema enforcement and schema evolution in batch modes (for example, directory listing or file notification) to land data into Unity Catalog-governed tables.

**Auto Loader** - A scalable, incremental file ingestion tool built on Structured Streaming that processes new files as they arrive in cloud storage, with schema inference, schema enforcement, and schema evolution. (source: https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/)

**Directory listing vs file notification** - The two file discovery modes: directory listing incrementally lists storage (no extra cloud permissions needed); file notification uses cloud event services (e.g., S3 event notifications) for faster, cheaper discovery at high file volume. (source: https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/)

In [0]:
al_root = f"{VOLUME}/autoloader"
al_src, al_meta = f"{al_root}/src", f"{al_root}/_meta"
dbutils.fs.rm(al_root, True)
spark.sql("DROP TABLE IF EXISTS autoloader_bronze")
write_json_lines(f"{al_src}/f1.json", [{"id": 1, "amount": 10.5}, {"id": 2, "amount": 20.0}])

def run_autoloader(hints="id BIGINT", evolution_mode="addNewColumns", file_notification="false"):
    q = (spark.readStream.format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", f"{al_meta}/schema")       # where the inferred schema is tracked
         .option("cloudFiles.schemaHints", hints)                        # schema ENFORCEMENT for chosen columns
         .option("cloudFiles.schemaEvolutionMode", evolution_mode)       # addNewColumns | rescue | failOnNewColumns | none
         .option("cloudFiles.useNotifications", file_notification)       # "false" = directory listing, "true" = file notification
         .load(al_src)
         .writeStream
         .option("checkpointLocation", f"{al_meta}/checkpoint")
         .option("mergeSchema", "true")
         .trigger(availableNow=True)                                     # batch-style incremental run
         .toTable("autoloader_bronze"))
    q.awaitTermination()

run_autoloader()
display(spark.table("autoloader_bronze"))

# Schema EVOLUTION: a new column appears in later files. With addNewColumns the stream normally stops
# once (UnknownFieldException) and resumes with the updated schema on the next trigger - a restart that
# Databricks recommends automating with Lakeflow Jobs. On serverless compute a stopped stream also fails
# the whole cell (a Python try/except cannot intercept it), so here the new column is instead declared up
# front with a schema hint and mergeSchema evolves the target table.
write_json_lines(f"{al_src}/f2.json", [{"id": 3, "amount": 5.0, "coupon": "SAVE10"}])
run_autoloader(hints="id BIGINT, coupon STRING")
display(spark.table("autoloader_bronze"))

# Schema ENFORCEMENT: a value that cannot be cast to the enforced type is kept in _rescued_data, not lost
write_json_lines(f"{al_src}/f3.json", [{"id": "not-a-number", "amount": 1.0, "coupon": "X"}])
run_autoloader(hints="id BIGINT, coupon STRING")   # keep the hint: it is not persisted in the schema location
display(spark.table("autoloader_bronze").filter("_rescued_data IS NOT NULL"))

# File notification mode (event-driven discovery) is the same code with file_notification="true";
# it needs cloud permissions to create queue/event subscriptions, so it is not executed here.

#### Objective 2.4
> Configure Lakeflow Connect to reliably ingest data from diverse enterprise sources into Unity Catalog-governed tables.

**Lakeflow Connect** - Databricks' suite of ingestion connectors covering local files, enterprise applications, databases, cloud storage, and message buses — from fully-managed connectors to fully-customizable frameworks — all landing data into Unity Catalog-governed tables. (source: https://docs.databricks.com/aws/en/ingestion/overview/)

**Managed connector architecture** - SaaS and file connectors consist of a Unity Catalog connection, an ingestion pipeline, and destination tables; database connectors add an ingestion gateway and staging storage to support continuous change capture (CDC). Each pipeline schedule is automatically created as a Lakeflow Job. (source: https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/)

In [0]:
# Lakeflow Connect managed connectors = a UC *connection* (credentials) + an *ingestion pipeline* (what to copy where).
w = try_run("databricks-sdk", get_workspace_client)
connections = []
if w:
    connections = try_run("list UC connections", lambda: list(w.connections.list())) or []
    print(f"Unity Catalog connections visible to you: {len(connections)}")
    for c in connections:
        print(" -", c.name, "|", c.connection_type)

# Template of an ingestion pipeline definition (managed connector). Needs a real connection to create.
ingestion_pipeline_spec = {
    "name": "lakeflow_connect_demo",
    "ingestion_definition": {
        "connection_name": connections[0].name if connections else "<your_uc_connection>",
        "objects": [
            {"table": {"source_schema": "dbo", "source_table": "orders",
                       "destination_catalog": CATALOG, "destination_schema": SCHEMA}},
            {"schema": {"source_schema": "sales",
                        "destination_catalog": CATALOG, "destination_schema": SCHEMA}},
        ],
    },
}

def validate_ingestion_spec(spec):
    d = spec["ingestion_definition"]
    assert d["connection_name"], "a UC connection is required"
    assert d["objects"], "choose at least one table or schema to ingest"
    for o in d["objects"]:
        kind = "table" if "table" in o else "schema"
        assert o[kind]["destination_catalog"] and o[kind]["destination_schema"], "destination must be a UC catalog.schema"
    return "spec looks valid"

print(validate_ingestion_spec(ingestion_pipeline_spec))
show_spec(ingestion_pipeline_spec)
# Reliability features you configure/observe: incremental (CDC) loads, automatic retries, schema-change handling,
# pipeline event log, and scheduling the pipeline with a Lakeflow Job.

#### Objective 2.5
> Use JDBC/ODBC or REST clients in notebooks to land data into cloud storage or directly into Unity Catalog-governed tables, usually orchestrated and scheduled with Lakeflow Jobs.

**JDBC/ODBC** - Industry-standard specifications for database connectivity. Databricks provides native JDBC and ODBC drivers to connect tools and applications (BI tools, IDEs, custom apps) to Databricks clusters and SQL warehouses. (source: https://docs.databricks.com/aws/en/integrations/jdbc-odbc-bi/)

**Service principal** - A specialized identity designed for automation and programmatic access to Databricks resources via APIs, without relying on individual user credentials; the typical identity for REST-based ingestion orchestrated by jobs. (source: https://docs.databricks.com/aws/en/admin/users-groups/service-principals/)

In [0]:
# --- REST: call an API, land the raw response in a Volume, then load it into a table --------------
# Demo API = this workspace's own Jobs API through the SDK's authenticated client.
w = get_workspace_client()
payload = try_run("REST call", lambda: w.api_client.do("GET", "/api/2.1/jobs/list", query={"limit": 25})) or {}
records = payload.get("jobs") or [{"note": "no jobs in this workspace"}]
rest_dir = f"{VOLUME}/rest_landing"
dbutils.fs.rm(rest_dir, True)
write_json_lines(f"{rest_dir}/jobs_{int(time.time())}.json", records)

(spark.read.json(rest_dir)
      .withColumn("_ingested_at", F.current_timestamp())
      .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("rest_jobs_bronze"))
display(spark.table("rest_jobs_bronze").limit(10))

# --- JDBC: parallel read from a relational database (pattern; needs a reachable database) ---------
def read_jdbc(url, table, scope, user_key, password_key,
              partition_col=None, lower=None, upper=None, num_partitions=8):
    reader = (spark.read.format("jdbc")
              .option("url", url)
              .option("dbtable", table)
              .option("user", dbutils.secrets.get(scope, user_key))          # never hard-code credentials
              .option("password", dbutils.secrets.get(scope, password_key)))
    if partition_col:                                                        # parallel reads by numeric range
        reader = (reader.option("partitionColumn", partition_col)
                        .option("lowerBound", lower).option("upperBound", upper)
                        .option("numPartitions", num_partitions))
    return reader.load()

# Example (uncomment with a real database + secret scope):
# df = read_jdbc("jdbc:postgresql://host:5432/db", "public.orders", "my_scope", "db_user", "db_pwd",
#                partition_col="order_id", lower=1, upper=1_000_000)
# df.write.mode("append").saveAsTable("orders_bronze")        # land directly into a UC table
# df.write.mode("append").parquet(f"{VOLUME}/orders_raw/")    # ...or into cloud storage / a Volume
# Schedule this notebook as a Lakeflow Jobs notebook task (see Section 4).

#### Objective 2.6
> Prioritize between Auto Loader, Lakeflow Connect (standard and managed connectors), partner connectors, and other ingestion methods based on technical requirements such as data volume, ingestion frequency, data types, and governance needs with Unity Catalog.

**Choosing an ingestion tool** - COPY INTO is the simple, idempotent choice for small or infrequent batches; Databricks recommends streaming tables/Auto Loader for more scalable and robust file ingestion; managed connectors handle SaaS apps and databases; partner connectors cover specialized third-party tools. (source: https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/copy-into/)

**Partner Connect** - The Databricks UI for discovering and integrating technology-partner solutions: create trial accounts with selected partners and connect your workspace to their tools for ingestion, transformation, BI/visualization, ML, and data quality. (source: https://docs.databricks.com/aws/en/lakehouse-architecture/interoperability-and-usability/best-practices/)

In [0]:
def recommend_ingestion(source, volume_gb_per_day, frequency, managed_connector_exists=False, uses_unity_catalog=True):
    gov = " Land into UC-governed tables/volumes for lineage and access control." if uses_unity_catalog else ""
    if source == "cloud_files":
        if volume_gb_per_day >= 1 or frequency in ("continuous", "hourly"):
            return ("Auto Loader", "Scalable, incremental file discovery with schema inference/evolution." + gov)
        return ("COPY INTO", "Simple, idempotent SQL loads for small / infrequent file batches." + gov)
    if source in ("saas_app", "database"):
        if managed_connector_exists:
            return ("Lakeflow Connect managed connector", "Fully managed, incremental/CDC ingestion, no custom code." + gov)
        return ("JDBC / REST in a Lakeflow Job" if source == "database" else "REST client in a Lakeflow Job",
                "No managed connector available -> custom code, orchestrated by Lakeflow Jobs." + gov)
    if source == "message_bus":
        return ("Structured Streaming / Lakeflow standard connector (e.g. Kafka)", "Low-latency continuous ingestion." + gov)
    if source == "partner_tool":
        return ("Partner connector (Partner Connect)", "Reuse an existing standardized ELT tool landing into Delta." + gov)
    if source == "local_file":
        return ("UI upload / local file import", "One-off, small files only.")
    return ("Review requirements", "Unknown source type.")

scenarios = [
    ("cloud_files", 500.0, "continuous", False),
    ("cloud_files", 0.1, "daily", False),
    ("database", 50.0, "hourly", True),
    ("database", 5.0, "daily", False),
    ("saas_app", 2.0, "daily", True),
    ("message_bus", 200.0, "continuous", False),
    ("local_file", 0.001, "once", False),
    ("partner_tool", 20.0, "daily", False),
]
rows = [(s, v, f, m) + recommend_ingestion(s, v, f, m) for s, v, f, m in scenarios]
display(spark.createDataFrame(
    rows, "source STRING, gb_per_day DOUBLE, frequency STRING, managed_connector_exists BOOLEAN, recommendation STRING, why STRING"))

#### Objective 2.7
> Ingest semi-structured and unstructured data (for example, JSON and nested data) via Lakeflow Connect and other managed connectors into Unity Catalog-governed Delta tables.

**Semi-structured data** - Data that does not fit a rigid tabular schema but still carries structure, such as JSON with nested objects and arrays (unstructured data — free text, images, audio — has no schema at all). Spark represents nested data with struct/array/map types, or with the VARIANT type below. (source: https://docs.databricks.com/aws/en/semi-structured/)

**VARIANT** - Databricks' standard SQL type for semi-structured data (Runtime 15.3+): an optimized encoding of JSON/XML that outperforms JSON strings for reads and writes, ingested with parse_json and stored in Delta tables. (source: https://docs.databricks.com/aws/en/ingestion/variant/)

In [0]:
# --- Semi-structured: nested JSON -> typed struct/array columns -> flat table ---------------------
json_rows = [
    json.dumps({"order_id": 1, "customer": {"id": "c1", "address": {"city": "Wroclaw", "zip": "50-001"}},
                "items": [{"sku": "A", "qty": 2}, {"sku": "B", "qty": 1}]}),
    json.dumps({"order_id": 2, "customer": {"id": "c2", "address": {"city": "Warsaw", "zip": "00-001"}},
                "items": [{"sku": "C", "qty": 5}]}),
]
raw = spark.createDataFrame([(r,) for r in json_rows], "raw STRING")
schema = ("order_id INT, customer STRUCT<id: STRING, address: STRUCT<city: STRING, zip: STRING>>, "
          "items ARRAY<STRUCT<sku: STRING, qty: INT>>")
parsed = raw.select(F.from_json("raw", schema).alias("j")).select("j.*")
parsed.printSchema()

flat = (parsed
        .select("order_id", F.col("customer.id").alias("customer_id"),
                F.col("customer.address.city").alias("city"), F.explode("items").alias("item"))
        .select("order_id", "customer_id", "city", "item.sku", "item.qty"))
flat.write.mode("overwrite").saveAsTable("orders_flat")           # Delta table in Unity Catalog
display(spark.table("orders_flat"))

# Schema discovery for unknown JSON
display(spark.sql(f"SELECT schema_of_json('{json_rows[0]}') AS inferred_schema"))

# VARIANT: flexible semi-structured type (needs a recent runtime / serverless)
raw.createOrReplaceTempView("raw_orders")
try_run("VARIANT", lambda: display(spark.sql("""
    SELECT v:order_id::int AS order_id, v:customer.address.city::string AS city, v:items[0].sku::string AS first_sku
    FROM (SELECT parse_json(raw) AS v FROM raw_orders)
""")))

# --- Unstructured: files read as binary (images, PDFs, audio ...) with metadata ---------------------
unstructured_dir = f"{VOLUME}/unstructured"
dbutils.fs.rm(unstructured_dir, True)
dbutils.fs.put(f"{unstructured_dir}/note.txt", "free text document", True)
dbutils.fs.put(f"{unstructured_dir}/readme.md", "# another document", True)
docs = spark.read.format("binaryFile").load(unstructured_dir)
display(docs.select("path", "length", "modificationTime", F.length("content").alias("bytes")))

---
## Section 3: Data Transformation and Modeling (22%)

#### Objective 3.1
> Implement data cleaning by reading bronze tables with PySpark/SQL, cleaning nulls, standardizing data types, and writing to new silver tables.

**Medallion architecture (Bronze / Silver / Gold)** - A data design pattern that organizes data into layers of increasing quality: Bronze ingests raw data with no cleanup, Silver performs data cleaning and validation, and Gold holds business-ready dimensional models and aggregates for BI and analytics teams. Also called a multi-hop architecture. (source: https://docs.databricks.com/aws/en/lakehouse/medallion/)

In [0]:
# Bronze = raw, everything as strings, with dirty values
raw = [
    (1, "  Ada ",  "2026-11-01", "10.50", "PL"),
    (2, None,      "2026-11-02", "abc",   "pl"),     # null name, bad amount
    (3, "Grace",   "not-a-date", None,    None),     # bad date, null amount/country
    (3, "Grace",   "not-a-date", None,    None),     # exact duplicate
    (4, "Linus",   "2026-11-04", "7",     "DE"),
]
(spark.createDataFrame(raw, "order_id INT, customer STRING, order_date STRING, amount STRING, country STRING")
      .write.mode("overwrite").saveAsTable("orders_bronze"))

# Silver = cleaned + typed. try_cast returns NULL instead of failing on bad values (safe under ANSI mode).
silver = (spark.table("orders_bronze")
          .withColumn("customer", F.trim("customer"))
          .withColumn("order_date", F.expr("try_cast(order_date AS DATE)"))
          .withColumn("amount", F.coalesce(F.expr("try_cast(amount AS DECIMAL(10,2))"), F.lit(0).cast("decimal(10,2)")))
          .withColumn("country", F.upper("country"))
          .na.fill({"customer": "UNKNOWN", "country": "N/A"})
          .dropna(subset=["order_date"])                 # rows without a valid date are not usable
          .dropDuplicates(["order_id"]))

silver.write.mode("overwrite").saveAsTable("orders_silver")
print("bronze rows:", spark.table("orders_bronze").count(), "| silver rows:", spark.table("orders_silver").count())
display(spark.table("orders_silver").orderBy("order_id"))
spark.table("orders_silver").printSchema()

# Same idea in SQL
display(spark.sql("""
    SELECT order_id, trim(customer) AS customer, try_cast(order_date AS DATE) AS order_date
    FROM orders_bronze WHERE try_cast(order_date AS DATE) IS NOT NULL
"""))

#### Objective 3.2
> Combine DataFrames with operations such as Inner join, left join, broadcast join, multiple keys, cross join, union, and union all.

**Join types** - An inner join keeps only rows matching on both sides; a left (outer) join keeps all left-table rows with nulls for non-matches; a cross join returns the Cartesian product of both DataFrames; multiple keys are joined on a list of columns. (source: https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-join.html)

**Broadcast join** - An optimization where the smaller side is sent in full to every executor, avoiding a shuffle; Spark does this automatically when a table is smaller than spark.sql.autoBroadcastJoinThreshold (default 10 MB). (source: https://spark.apache.org/docs/latest/sql-performance-tuning.html)

**Union vs unionAll** - In the PySpark DataFrame API, union() stacks DataFrames positionally and does NOT remove duplicates — it is equivalent to UNION ALL in SQL (unionAll is a deprecated alias); deduplication requires an explicit .distinct(). (source: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.union.html)

In [0]:
customers = spark.createDataFrame([(1, "Ada", "PL"), (2, "Grace", "US"), (3, "Linus", "FI")],
                                  "customer_id INT, name STRING, country STRING")
orders = spark.createDataFrame([(100, 1, "PL", 10.0), (101, 2, "US", 20.0), (102, 4, "DE", 5.0)],
                               "order_id INT, customer_id INT, country STRING, amount DOUBLE")

print("INNER JOIN");  display(orders.join(customers, "customer_id", "inner"))
print("LEFT JOIN");   display(orders.join(customers, "customer_id", "left"))
print("BROADCAST JOIN (small side shipped to every executor, avoids a shuffle)")
bj = orders.join(F.broadcast(customers), "customer_id")
display(bj)
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    bj.explain()
print("plan uses BroadcastHashJoin:", "BroadcastHashJoin" in buf.getvalue())

print("MULTIPLE KEYS")
display(orders.join(customers, ["customer_id", "country"], "inner"))

print("CROSS JOIN (every combination)")
sizes = spark.createDataFrame([("S",), ("M",)], "size STRING")
display(customers.select("name").crossJoin(sizes))

print("UNION / UNION ALL")
more_customers = spark.createDataFrame([(3, "Linus", "FI"), (4, "Margaret", "US")], customers.schema)
display(customers.union(more_customers))                  # DataFrame.union == SQL UNION ALL (keeps duplicates)
display(customers.unionAll(more_customers))               # alias of union
display(customers.union(more_customers).distinct())       # SQL UNION (removes duplicates)
reordered = more_customers.select("name", "country", "customer_id")
display(customers.unionByName(reordered))                 # match columns by name instead of position

#### Objective 3.3
> Manipulate columns, rows, and table structures by adding, dropping, splitting, renaming column names, applying filters, and exploding arrays.

**explode** - A generator function that produces a new row for each element in an array or map column, flattening nested collections (e.g., an array of item structs) into flat rows; explode_outer also keeps null/empty rows. (source: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.explode.html)

**split** - Splits a string column around a regex pattern into an array column — commonly paired with column indexing to separate e.g. full_name into first and last name. (source: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.split.html)

In [0]:
df = spark.createDataFrame(
    [(1, "Ada Lovelace", ["python", "sql"], "tmp"), (2, "Grace Hopper", ["cobol"], "tmp"), (-1, "Bad Row", [], "tmp")],
    "id INT, full_name STRING, skills ARRAY<STRING>, junk STRING")

out = (df
       .filter(F.col("id") > 0)                                         # filter rows
       .withColumn("first_name", F.split("full_name", " ")[0])          # add + split
       .withColumn("last_name", F.split("full_name", " ")[1])
       .drop("junk")                                                    # drop column
       .withColumnRenamed("skills", "skill_list")                       # rename column
       .withColumn("skill", F.explode("skill_list")))                   # explode array -> one row per element
display(out)
display(df.selectExpr("id", "upper(full_name) AS name_upper", "size(skills) AS n_skills"))

# Table structure changes on a Delta table (rename/drop need column mapping)
spark.sql("""
    CREATE OR REPLACE TABLE struct_demo (id INT, junk STRING)
    TBLPROPERTIES ('delta.columnMapping.mode' = 'name', 'delta.minReaderVersion' = '2', 'delta.minWriterVersion' = '5')
""")
spark.sql("ALTER TABLE struct_demo ADD COLUMNS (amount DOUBLE)")
spark.sql("ALTER TABLE struct_demo RENAME COLUMN junk TO note")
spark.sql("ALTER TABLE struct_demo DROP COLUMN note")
spark.table("struct_demo").printSchema()

#### Objective 3.4
> Perform data deduplication operations and aggregate operations on DataFrames, such as count, approximate count distinct, and mean, summary.

**Deduplication** - dropDuplicates() returns a new DataFrame with duplicate rows removed, optionally considering only a subset of columns to define duplicates; keeping the latest record per key is typically done with a Window ordered by a timestamp. (source: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.dropDuplicates.html)

**approx_count_distinct** - An aggregate function that estimates the number of distinct values using HyperLogLog sketches, trading bounded error for large speedups over exact COUNT(DISTINCT) on big data. (source: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.approx_count_distinct.html)

**summary** - Computes count, mean, stddev, min, approximate quartiles/percentiles, and max for numeric and string columns in a single call. (source: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.summary.html)

In [0]:
sales = spark.createDataFrame(
    [(1, "A", 10.0, "2026-11-01"), (1, "A", 10.0, "2026-11-01"),
     (2, "B", 20.0, "2026-11-02"), (2, "B", 25.0, "2026-11-03"), (3, "A", 5.0, "2026-11-04")],
    "id INT, product STRING, amount DOUBLE, ts STRING")

print("Exact duplicates removed:");                    display(sales.dropDuplicates())
print("One row per id (arbitrary survivor):");         display(sales.dropDuplicates(["id"]))

w = Window.partitionBy("id").orderBy(F.col("ts").desc())
print("Deterministic: keep the latest row per id:")
display(sales.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn"))

print("Aggregations:")
display(sales.groupBy("product").agg(
    F.count("*").alias("n_rows"),
    F.countDistinct("id").alias("exact_distinct_ids"),
    F.approx_count_distinct("id").alias("approx_distinct_ids"),   # HyperLogLog++, far cheaper on big data
    F.mean("amount").alias("mean_amount"),
    F.sum("amount").alias("total")))

print("summary() = count, mean, stddev, min, quartiles, max");  display(sales.summary())
print("describe() = count, mean, stddev, min, max");            display(sales.describe("amount"))

#### Objective 3.5
> Understand the basic tuning parameters (spark.sql.shuffle.partitions, spark.default.parallelism, spark.executor/driver.memory, spark.sql.autoBroadcastJoinThreshold) and re-measure the performance.

**Shuffle and spark.sql.shuffle.partitions** - Wide transformations (joins, groupBy) force a shuffle — a network data exchange between executors; the number of shuffle partitions (default 200) sets the parallelism of that exchange and can be tuned per query. (source: https://spark.apache.org/docs/latest/configuration.html)

**spark.sql.autoBroadcastJoinThreshold** - The size threshold (default 10 MB) below which Spark broadcasts a table to executors for a broadcast join instead of shuffling it. (source: https://spark.apache.org/docs/latest/sql-performance-tuning.html)

**Executor / driver memory** - Heap memory for the JVM processes that run tasks (executors) and for the driver that coordinates the job and holds collect()-ed results; set at cluster creation time, not from notebooks. (source: https://spark.apache.org/docs/latest/configuration.html)

In [0]:
params = ["spark.sql.shuffle.partitions", "spark.default.parallelism", "spark.executor.memory",
          "spark.driver.memory", "spark.sql.autoBroadcastJoinThreshold", "spark.sql.adaptive.enabled"]
print("Current values (None = not exposed on this compute, typical for serverless):")
for p in params:
    print(f"  {p:<42} = {conf(p)}")

def timed(label, df):
    t0 = time.time()
    df.write.format("noop").mode("overwrite").save()          # run the full query without writing data
    print(f"  {label:<34}{time.time() - t0:6.2f}s")

def workload():
    return (spark.range(0, 20_000_000).withColumn("k", F.col("id") % 1000)
            .groupBy("k").agg(F.sum("id").alias("s")))

originals = {k: conf(k) for k in ["spark.sql.shuffle.partitions", "spark.sql.adaptive.enabled",
                                  "spark.sql.autoBroadcastJoinThreshold"]}

# 1) shuffle partitions: too few -> big tasks / spill, too many -> scheduling overhead. Disable AQE to see the raw effect.
set_conf("spark.sql.adaptive.enabled", "false")
print("Group-by with different spark.sql.shuffle.partitions:")
for n in (2, 8, 200, 2000):
    if set_conf("spark.sql.shuffle.partitions", str(n)):
        timed(f"shuffle.partitions = {n}", workload())

# 2) autoBroadcastJoinThreshold: -1 disables automatic broadcast joins
big = spark.range(0, 5_000_000).withColumn("k", F.col("id") % 1000)
small = spark.range(0, 1000).withColumnRenamed("id", "k")
def join_strategy(threshold):
    set_conf("spark.sql.autoBroadcastJoinThreshold", str(threshold))
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        big.join(small, "k").explain()
    return "BroadcastHashJoin" if "BroadcastHashJoin" in buf.getvalue() else "SortMergeJoin"
print("threshold -1 (disabled):", join_strategy(-1))
print("threshold 10 MB        :", join_strategy(10 * 1024 * 1024))

# Restore whatever was set before
for k, v in originals.items():
    if v is not None:
        set_conf(k, v)

# 3) spark.default.parallelism / executor & driver memory are cluster-level (static): set them in the cluster's
#    Spark config and restart; they cannot be changed from a running notebook. Re-measure after each change.

#### Objective 3.6
> Understand the difference between, and how to build, Gold layer objects such as materialized views, views, streaming tables, and tables for BI and analytics teams in Unity Catalog.

**Materialized view** - A pre-computed view using batch semantics: results are stored and refreshed incrementally on a schedule, and are automatically recomputed when source data changes (joins recompute when dimension tables change). (source: https://docs.databricks.com/aws/en/data-engineering/tables-views/)

**Streaming table** - A Unity Catalog managed table that includes its processing logic via flows: it processes each row only once from append-only sources and refreshes with REFRESH (incremental) or REFRESH ... FULL (reprocess everything). (source: https://docs.databricks.com/aws/en/ldp/concepts/streaming-tables/)

**View vs table** - A view is a saved query with no storage of its own, evaluated on every access; a table physically stores data. Materialized views and streaming tables sit in between: stored, but automatically maintained from their source query. (source: https://docs.databricks.com/aws/en/data-engineering/tables-views/)

In [0]:
# Small silver dataset so this cell is self-contained
spark.createDataFrame([("2026-11-01", 10.0), ("2026-11-01", 15.0), ("2026-11-02", 20.0)], "order_date STRING, amount DOUBLE") \
     .write.mode("overwrite").saveAsTable("gold_src_silver")
spark.sql("CREATE OR REPLACE TABLE gold_st_source (event_id INT, payload STRING)")
spark.sql("INSERT INTO gold_st_source VALUES (1, 'a'), (2, 'b')")

# TABLE: physically stored result, you decide when it is rebuilt
spark.sql("""CREATE OR REPLACE TABLE gold_sales_daily AS
             SELECT order_date, count(*) AS orders, sum(amount) AS revenue FROM gold_src_silver GROUP BY order_date""")
# VIEW: just a saved query, always evaluated at read time (no storage, no precompute)
spark.sql("CREATE OR REPLACE VIEW gold_sales_view AS SELECT * FROM gold_sales_daily WHERE revenue > 12")
# MATERIALIZED VIEW: precomputed result kept up to date by a refresh (runs on serverless / SQL warehouse / pipeline)
try_run("MATERIALIZED VIEW", lambda: spark.sql("""CREATE OR REPLACE MATERIALIZED VIEW gold_sales_mv AS
             SELECT order_date, sum(amount) AS revenue FROM gold_src_silver GROUP BY order_date"""))
# STREAMING TABLE: incrementally ingests an append-only source, processes each row once
try_run("STREAMING TABLE", lambda: spark.sql(
    "CREATE OR REFRESH STREAMING TABLE gold_events_st AS SELECT * FROM STREAM(gold_st_source)"))

display(spark.sql(f"SHOW TABLES IN `{CATALOG}`.`{SCHEMA}` LIKE 'gold_*'"))

display(spark.createDataFrame([
    ("Table", "Yes (Delta)", "You run the write/MERGE", "Full control, simple, any compute", "Reporting layers you rebuild on a schedule"),
    ("View", "No", "Evaluated on every query", "Always fresh, cost paid at read time", "Light logic, security layers, abstraction"),
    ("Materialized view", "Yes (managed)", "Refresh; incremental when possible", "Fast BI reads, SQL defined", "Aggregates dashboards query repeatedly"),
    ("Streaming table", "Yes (managed)", "Incremental, exactly-once per source row", "Append-only sources, low latency", "Ingestion / append-only silver or gold"),
], "object STRING, stores_data STRING, refresh_behavior STRING, characteristic STRING, typical_use STRING"))

# The same objects inside a Lakeflow Declarative Pipeline (Python) - shown, not executed here:
print('''
from pyspark import pipelines as dp
from pyspark.sql import functions as F

@dp.table                       # streaming table (function returns a streaming DataFrame)
def events_st():
    return spark.readStream.table("catalog.schema.gold_st_source")

@dp.materialized_view           # materialized view (function returns a batch DataFrame)
def sales_daily_mv():
    return spark.read.table("catalog.schema.silver_orders").groupBy("order_date").agg(F.sum("amount").alias("revenue"))
''')

#### Objective 3.7
> Apply data quality checks and validation rules to ensure reliable Silver and Gold datasets.

**Data quality expectations** - True/false SQL expressions applied to every row in a Lakeflow pipeline dataset; on violation the pipeline can warn (expect), drop the record (expect_or_drop), or fail the run (expect_or_fail), and metrics are always emitted to the pipeline event log for trend analysis. (source: https://docs.databricks.com/aws/en/ldp/best-practices/)

In [0]:
df = spark.createDataFrame([(1, 10.0, "a@x.com"), (2, -5.0, "bad-email"), (3, None, "c@x.com")],
                           "id INT, amount DOUBLE, email STRING")

rules = {
    "amount_not_null": "amount IS NOT NULL",
    "amount_positive": "amount > 0",
    "email_valid": "email RLIKE '^[^@]+@[^@]+[.][a-z]+$'",
}
checked = df
for name, cond in rules.items():
    checked = checked.withColumn(f"ok_{name}", F.coalesce(F.expr(cond), F.lit(False)))
all_ok = " AND ".join(f"ok_{n}" for n in rules)

valid = checked.filter(all_ok)
quarantine = checked.filter(f"NOT ({all_ok})")
print("valid rows:");                           display(valid.select("id", "amount", "email"))
print("quarantined rows (route to a rejects table):"); display(quarantine)

print("Failure count per rule:")
display(checked.agg(*[F.sum((~F.col(f"ok_{n}")).cast("int")).alias(n) for n in rules]))

# Enforce rules in the table itself with Delta constraints
spark.sql("CREATE OR REPLACE TABLE silver_checked (id INT NOT NULL, amount DOUBLE)")
spark.sql("ALTER TABLE silver_checked ADD CONSTRAINT amount_positive CHECK (amount > 0)")
try:
    spark.sql("INSERT INTO silver_checked VALUES (1, -1.0)")
except Exception as e:
    print("Rejected by CHECK constraint ->", type(e).__name__)
spark.sql("INSERT INTO silver_checked VALUES (2, 5.0)")
display(spark.table("silver_checked"))

# Declarative equivalent inside a Lakeflow Declarative Pipeline (expectations) - shown, not executed:
print('''
@dp.table
@dp.expect("has_id", "id IS NOT NULL")                  # warn: keep row, record the violation metric
@dp.expect_or_drop("valid_amount", "amount > 0")        # drop invalid rows
@dp.expect_or_fail("valid_date", "order_date IS NOT NULL")  # fail the update
def silver_orders():
    return spark.readStream.table("bronze_orders")
''')

---
## Section 4: Working with Lakeflow Jobs (16%)

#### Objective 4.1
> Implement control flows (retries and conditional tasks such as branching and looping) using Lakeflow Jobs for pipeline orchestration

**Lakeflow Jobs** - Workflow automation for Databricks: orchestrate and schedule data processing workloads by coordinating multiple tasks (notebooks, SQL, pipelines) as part of a larger workflow, with control-flow logic such as branching (if/else) and looping (for each) in a DAG. (source: https://docs.databricks.com/aws/en/jobs/)

**Retries and Run if (conditional) tasks** - Retries re-run a failed task up to a configured number of times (errors are often transient and resolved through restart); Run if tasks run conditionally based on the outcome of upstream tasks (all succeeded, at least one succeeded, none failed, all done, at least one failed, all failed). (source: https://docs.databricks.com/aws/en/jobs/control-flow/)

In [0]:
NB = notebook_path()

def nb_task(key, depends_on=None, params=None, **extra):
    t = {"task_key": key, "notebook_task": {"notebook_path": NB, "base_parameters": params or {}}}
    if depends_on:
        t["depends_on"] = depends_on
    t.update(extra)
    return t

control_flow_job = {
    "name": "demo_control_flow",
    "tasks": [
        # RETRIES: re-run a failing task automatically
        nb_task("extract", params={"step": "extract"},
                max_retries=3, min_retry_interval_millis=60000, retry_on_timeout=True, timeout_seconds=1800),
        # BRANCHING: If/else condition task reading a task value
        {"task_key": "has_rows", "depends_on": [{"task_key": "extract"}],
         "condition_task": {"op": "GREATER_THAN", "left": "{{tasks.extract.values.row_count}}", "right": "0"}},
        nb_task("transform", depends_on=[{"task_key": "has_rows", "outcome": "true"}], params={"step": "transform"}),
        nb_task("alert_empty", depends_on=[{"task_key": "has_rows", "outcome": "false"}], params={"step": "alert"}),
        # LOOPING: For each task fans out over a list of inputs
        {"task_key": "per_country", "depends_on": [{"task_key": "transform"}],
         "for_each_task": {"inputs": json.dumps(["PL", "DE", "US"]), "concurrency": 2,
                           "task": nb_task("per_country_iteration", params={"country": "{{input}}"})}},
        # RUN-IF: run cleanup no matter what happened upstream
        nb_task("always_cleanup", depends_on=[{"task_key": "per_country"}, {"task_key": "alert_empty"}],
                params={"step": "cleanup"}, run_if="ALL_DONE"),
    ],
}
show_spec(control_flow_job)
create_job_if_allowed(control_flow_job)

# What a retry does, simulated locally with exponential back-off
def with_retries(fn, max_retries=3, base_delay=0.2):
    for attempt in range(max_retries + 1):
        try:
            return fn()
        except Exception as e:
            if attempt == max_retries:
                raise
            delay = base_delay * 2 ** attempt
            print(f"  attempt {attempt + 1} failed ({e}); retrying in {delay:.1f}s")
            time.sleep(delay)

state = {"calls": 0}
def flaky_extract():
    state["calls"] += 1
    if state["calls"] < 3:
        raise RuntimeError("transient error")
    return "ok"

print("result:", with_retries(flaky_extract), "after", state["calls"], "calls")

# Task values pass data between tasks (the condition above reads row_count). Outside a job, use debugValue to test.
try_run("taskValues.set", lambda: dbutils.jobs.taskValues.set(key="row_count", value=42))
print("row_count seen by a downstream task:",
      try_run("taskValues.get", lambda: dbutils.jobs.taskValues.get(taskKey="extract", key="row_count", default=0, debugValue=42)))

#### Objective 4.2
> Configure common tasks (notebook, SQL query, dashboard, and pipeline tasks) and their dependencies using Lakeflow Jobs and its DAG-based task graph

**Job tasks and dependencies** - A job runs one or more tasks — notebook, SQL query, dashboard refresh, pipeline update, and more — with configurable dependencies between tasks, forming a DAG-based task graph that determines execution order. (source: https://docs.databricks.com/aws/en/jobs/)

**DAG (directed acyclic graph)** - The task dependency structure: nodes are tasks, edges are dependencies; a task runs only once all its dependencies complete, so an upstream failure blocks its downstream tasks — visible in the jobs timeline view. (source: https://docs.databricks.com/aws/en/jobs/monitor/)

In [0]:
from graphlib import TopologicalSorter

dag_job = {
    "name": "demo_task_types",
    "tasks": [
        {"task_key": "ingest", "notebook_task": {"notebook_path": NB}},
        {"task_key": "silver_pipeline", "depends_on": [{"task_key": "ingest"}],
         "pipeline_task": {"pipeline_id": "<lakeflow-pipeline-id>", "full_refresh": False}},
        {"task_key": "quality_sql", "depends_on": [{"task_key": "silver_pipeline"}],
         "sql_task": {"warehouse_id": "<sql-warehouse-id>", "query": {"query_id": "<saved-query-id>"}}},
        {"task_key": "gold_notebook", "depends_on": [{"task_key": "silver_pipeline"}],
         "notebook_task": {"notebook_path": NB}},
        {"task_key": "refresh_dashboard", "depends_on": [{"task_key": "quality_sql"}, {"task_key": "gold_notebook"}],
         "dashboard_task": {"dashboard_id": "<dashboard-id>", "warehouse_id": "<sql-warehouse-id>"}},
    ],
}
show_spec(dag_job)

# The task graph is a DAG: tasks whose dependencies are done run in parallel
graph = {t["task_key"]: {d["task_key"] for d in t.get("depends_on", [])} for t in dag_job["tasks"]}
ts = TopologicalSorter(graph)
ts.prepare()
wave = 1
while ts.is_active():
    ready = sorted(ts.get_ready())
    print(f"wave {wave}: run in parallel -> {ready}")
    ts.done(*ready)
    wave += 1

task_type = {t["task_key"]: next(k for k in t if k.endswith("_task")) for t in dag_job["tasks"]}
edges = [(up, t["task_key"]) for t in dag_job["tasks"] for up in [d["task_key"] for d in t.get("depends_on", [])]]
display(spark.createDataFrame([(a, b, task_type[b]) for a, b in edges], "upstream STRING, downstream STRING, downstream_task_type STRING"))

try:                                    # a cycle is not a valid DAG
    list(TopologicalSorter({"a": {"b"}, "b": {"a"}}).static_order())
except Exception as e:
    print("cycle rejected:", type(e).__name__)

#### Objective 4.3
> Implement job schedules using Lakeflow Jobs with an understanding of trigger types (scheduled, file arrival, and table update)

**Scheduled trigger** - A cron-expression schedule (with timezone) that runs the job on a fixed wall-clock cadence. (source: https://docs.databricks.com/aws/en/jobs/)

**File arrival trigger** - Starts a job when new files arrive at a monitored UC external location or volume (checked roughly every minute, recursively) — useful when a scheduled job's efficiency is compromised by irregular data arrivals. (source: https://docs.databricks.com/aws/en/jobs/file-arrival-triggers/)

**Table update trigger** - Starts a job when a monitored table is updated (including merges and deletes); works for managed and external Delta tables, materialized views, streaming tables, and views, and supports 'any' or 'all' tables semantics. (source: https://docs.databricks.com/aws/en/jobs/trigger-table-update/)

In [0]:
scheduled = {"schedule": {"quartz_cron_expression": "0 30 6 * * ?",       # every day 06:30
                          "timezone_id": "Europe/Warsaw", "pause_status": "UNPAUSED"}}

file_arrival = {"trigger": {"pause_status": "UNPAUSED",
                            "file_arrival": {"url": f"{VOLUME}/trigger_landing/",
                                             "min_time_between_triggers_seconds": 60,
                                             "wait_after_last_change_seconds": 30}}}

table_update = {"trigger": {"pause_status": "UNPAUSED",
                            "table_update": {"table_names": [f"{CATALOG}.{SCHEMA}.orders_bronze"],
                                             "condition": "ANY_UPDATED",
                                             "min_time_between_triggers_seconds": 60,
                                             "wait_after_last_change_seconds": 30}}}

continuous = {"continuous": {"pause_status": "UNPAUSED"}}

for name, cfg in [("SCHEDULED (time-based)", scheduled), ("FILE ARRIVAL", file_arrival),
                  ("TABLE UPDATE", table_update), ("CONTINUOUS", continuous)]:
    print(f"--- {name}")
    show_spec(cfg)

def check_quartz(expr):
    parts = expr.split()
    assert len(parts) in (6, 7), "Quartz cron needs 6-7 fields: sec min hour day-of-month month day-of-week [year]"
    assert "?" in (parts[3], parts[5]), "use '?' in either day-of-month or day-of-week"
    return "valid quartz expression"

print(check_quartz(scheduled["schedule"]["quartz_cron_expression"]))

# Complete job = tasks + exactly one trigger style. Printed only unless run_side_effects=true.
trigger_job = {"name": "demo_table_update_trigger", "tasks": [nb_task("refresh", params={"step": "refresh"})], **table_update}
create_job_if_allowed(trigger_job)

#### Objective 4.4
> Choose between time-based and data-driven triggers based on data availability and pipeline dependencies.

**Time-based vs data-driven triggers** - Time-based (cron) triggers run on a fixed schedule regardless of whether data is ready — simple, but they can run on empty or late data. Data-driven triggers (file arrival, table update) fire when upstream data is actually available, making them the better choice when a pipeline depends on data readiness rather than on the clock. (source: https://docs.databricks.com/aws/en/jobs/file-arrival-triggers/)

In [0]:
def choose_trigger(data_arrival, upstream_is_uc_table, needs_fixed_sla_time, latency_requirement):
    if latency_requirement == "seconds":
        return ("Continuous", "Always-on job for near-real-time processing.")
    if upstream_is_uc_table and data_arrival == "irregular":
        return ("Table update trigger", "Run whenever the upstream Unity Catalog table changes.")
    if data_arrival == "irregular" and not upstream_is_uc_table:
        return ("File arrival trigger", "Run when new files land in a Volume / external location.")
    if needs_fixed_sla_time or data_arrival == "predictable":
        return ("Scheduled (cron)", "Data arrives at predictable times or a fixed delivery time is required.")
    return ("Scheduled (cron)", "Default when there is no event to listen to.")

cases = [
    ("predictable", False, True,  "hours",   "Nightly export arrives at 02:00, report due 07:00"),
    ("irregular",   False, False, "minutes", "Partner drops CSV files at random times"),
    ("irregular",   True,  False, "minutes", "Gold refresh after upstream silver table is updated"),
    ("predictable", True,  False, "seconds", "Event stream that must be processed continuously"),
]
rows = [(desc, *choose_trigger(a, uc, sla, lat)) for a, uc, sla, lat, desc in cases]
display(spark.createDataFrame(rows, "scenario STRING, recommended_trigger STRING, why STRING"))

# Rule of thumb: time-based = simple, predictable cost, but may run on stale/missing data or waste runs.
#                data-driven = runs exactly when inputs change, removes idle polling and fixed-time guesswork.

---
## Section 5: Implementing CI/CD (10%)

#### Objective 5.1
> Manage your code development workflow within the Databricks workspace UI, including creating and switching between branches in Databricks Git folders (formerly Databricks Repos), committing and pushing changes, and creating pull requests using Databricks Git integration.

**Databricks Git folders (formerly Repos)** - A visual Git client and API integrated into the workspace: clone repositories, create and switch branches, commit and push, pull, visually compare diffs, resolve merge conflicts, and open pull requests — developing notebooks and files with standard Git version-control and CI/CD workflows. (source: https://docs.databricks.com/aws/en/repos/git-folders-concepts/)

In [0]:
w = get_workspace_client()

# Git folders visible in this workspace
repos = try_run("list Git folders", lambda: list(w.repos.list())) or []
print(f"Git folders: {len(repos)}")
display(spark.createDataFrame(
    [(str(r.id), r.path, r.url, r.branch, str(r.provider)) for r in repos],
    "id STRING, path STRING, url STRING, branch STRING, provider STRING"))

# The day-to-day workflow (UI actions; the Git folder UI wraps the git operations below)
workflow = [
    (1, "Create / clone a Git folder", "Workspace > Create > Git folder (paste repo URL)", "git clone"),
    (2, "Create a feature branch",     "Git folder menu > branch dropdown > Create branch", "git checkout -b feature/x"),
    (3, "Switch branches",             "Branch dropdown > pick branch", "git checkout <branch>"),
    (4, "Edit notebooks / files",      "Edit in the workspace", "(working tree changes)"),
    (5, "Commit & push",               "Git dialog > select changes > message > Commit & Push", "git add; git commit; git push"),
    (6, "Pull latest",                 "Git dialog > Pull", "git pull"),
    (7, "Create a pull request",       "Git dialog > Create PR (opens your Git provider)", "provider UI (GitHub / Azure DevOps ...)"),
]
display(spark.createDataFrame(workflow, "step INT, action STRING, where_in_databricks STRING, git_equivalent STRING"))

# Same operations through the API/SDK (only if you give a repo URL and enable side effects)
GIT_URL = ""      # e.g. "https://github.com/<org>/<repo>.git"
if RUN_SIDE_EFFECTS and GIT_URL:
    me = spark.sql("SELECT current_user()").first()[0]
    repo = w.repos.create(url=GIT_URL, provider="gitHub", path=f"/Repos/{me}/de_associate_demo")
    w.repos.update(repo_id=repo.id, branch="main")      # switch branch (the branch must exist remotely)
    print("created Git folder:", repo.path)
else:
    print("Set GIT_URL and run_side_effects=true to create a Git folder via the SDK. Commit/push/PR are done in the UI.")

#### Objective 5.2
> Understand environment-specific configuration using Declarative Automation Bundle (formerly Databricks Asset Bundles) variables and overrides while promoting the same codebase across dev, test, and prod targets.

**Declarative Automation Bundles (DABs)** - Databricks projects as code: a databricks.yml that defines variables, resources (jobs, pipelines, dashboards, ...), and per-environment targets, which you validate, deploy, and run programmatically. (source: https://docs.databricks.com/aws/en/dev-tools/cli/bundle-commands/)

**Variables and target overrides** - Custom variables are declared with defaults in the variables mapping; targets (dev/test/prod) can override them. At deploy time the CLI resolves values in order: --var options, BUNDLE_VAR_* environment variables, variable-overrides.json, target mappings, then declared defaults. (source: https://docs.databricks.com/aws/en/dev-tools/bundles/variables/)

In [0]:
import yaml

bundle_yaml = '''
bundle:
  name: de_associate_demo

variables:
  catalog:
    description: Unity Catalog catalog for this environment
    default: dev_catalog
  schedule_pause:
    description: Whether the schedule is active
    default: PAUSED

resources:
  jobs:
    nightly_etl:
      name: "[${bundle.target}] nightly_etl"
      schedule:
        quartz_cron_expression: "0 0 2 * * ?"
        timezone_id: Europe/Warsaw
        pause_status: ${var.schedule_pause}
      tasks:
        - task_key: etl
          notebook_task:
            notebook_path: ./src/etl.py
            base_parameters:
              catalog: ${var.catalog}

targets:
  dev:
    mode: development
    default: true
  test:
    variables:
      catalog: test_catalog
  prod:
    mode: production
    variables:
      catalog: prod_catalog
      schedule_pause: UNPAUSED
'''
cfg = yaml.safe_load(bundle_yaml)

def resolve(target):
    """Mimics bundle variable resolution: default < target override (CLI --var and BUNDLE_VAR_* beat both)."""
    values = {k: v.get("default") for k, v in cfg["variables"].items()}
    values.update(cfg["targets"][target].get("variables", {}))
    def sub(text):
        text = re.sub(r"\$\{var\.(\w+)\}", lambda m: str(values[m.group(1)]), str(text))
        return text.replace("${bundle.target}", target)
    job = cfg["resources"]["jobs"]["nightly_etl"]
    return (target, cfg["targets"][target].get("mode", "(none)"), sub(job["name"]),
            job["tasks"][0]["notebook_task"]["base_parameters"]["catalog"].replace("${var.catalog}", values["catalog"]),
            sub(job["schedule"]["pause_status"]))

display(spark.createDataFrame([resolve(t) for t in cfg["targets"]],
                              "target STRING, mode STRING, job_name STRING, catalog STRING, schedule_status STRING"))

print("Same code, different config per target. Override precedence (highest first):")
print("  1) databricks bundle deploy -t prod --var=\"catalog=other\"")
print("  2) environment variable BUNDLE_VAR_catalog=other")
print("  3) targets.<name>.variables in databricks.yml")
print("  4) variables.<name>.default in databricks.yml")

#### Objective 5.3
> Deploy Declarative Automation Bundles (formerly Databricks Asset Bundles) to package, configure, and promote Lakeflow Jobs, Lakeflow Declarative Pipelines, and other workspace assets across dev, test, and prod environments.

**Bundle deployment** - databricks bundle deploy -t <target> packages the configuration and source code and creates/updates the workspace resources (jobs, pipelines, ...) in the target environment, letting the same codebase be promoted across dev, test, and prod. (source: https://docs.databricks.com/aws/en/dev-tools/cli/bundle-commands/)

**Lakeflow Declarative Pipelines** - A declarative framework for batch and streaming pipelines in SQL and Python: you declare streaming tables, materialized views, and flows, and the pipeline provides automatic orchestration and incremental updates. (source: https://docs.databricks.com/aws/en/ldp/concepts/)

In [0]:
bundle_dir = "/tmp/de_associate_bundle"
shutil.rmtree(bundle_dir, ignore_errors=True)
os.makedirs(f"{bundle_dir}/src", exist_ok=True)

files = {
    "databricks.yml": '''
bundle:
  name: de_associate_demo
variables:
  catalog:
    default: dev_catalog
resources:
  pipelines:
    silver_pipeline:
      name: "[${bundle.target}] silver_pipeline"
      catalog: ${var.catalog}
      schema: silver
      serverless: true
      libraries:
        - file:
            path: ./src/silver_pipeline.py
  jobs:
    orchestrator:
      name: "[${bundle.target}] orchestrator"
      tasks:
        - task_key: refresh_silver
          pipeline_task:
            pipeline_id: ${resources.pipelines.silver_pipeline.id}
        - task_key: build_gold
          depends_on:
            - task_key: refresh_silver
          notebook_task:
            notebook_path: ./src/build_gold.py
targets:
  dev:
    mode: development
    default: true
  prod:
    mode: production
    variables:
      catalog: prod_catalog
''',
    "src/silver_pipeline.py": "from pyspark import pipelines as dp\n\n@dp.table\ndef silver():\n    return spark.read.table('bronze')\n",
    "src/build_gold.py": "# Databricks notebook source\nprint('build gold')\n",
}
for rel, content in files.items():
    with open(f"{bundle_dir}/{rel}", "w") as f:
        f.write(content)

parsed = yaml.safe_load(files["databricks.yml"])
assets = [(kind, name) for kind, items in parsed["resources"].items() for name in items]
display(spark.createDataFrame(assets, "resource_type STRING, resource_name STRING"))
print("Targets:", list(parsed["targets"]))

if shutil.which("databricks"):
    r = subprocess.run(["databricks", "bundle", "validate", "-t", "dev"], cwd=bundle_dir,
                       capture_output=True, text=True, timeout=120)
    print(r.stdout or r.stderr)
else:
    print("Databricks CLI not found on this compute; commands you would run from CI or a terminal:")

for cmd in ["databricks bundle validate -t dev",
            "databricks bundle deploy   -t dev      # package + create/update jobs & pipelines in dev",
            "databricks bundle run orchestrator -t dev",
            "databricks bundle deploy   -t test     # promote the SAME bundle to test",
            "databricks bundle deploy   -t prod     # ...then to prod (production mode)"]:
    print("  $", cmd)

#### Objective 5.4
> Understand the Databricks CLI to validate, deploy, and manage Declarative Automation Bundles (formerly Databricks Asset Bundles) and other workspace assets in automated CI/CD workflows.

**Databricks CLI** - The command-line tool for the Databricks APIs, usable both interactively and in automated CI/CD workflows; key bundle commands: init (scaffold from a template), validate (check config and references), deploy -t <target> (create/update resources), run, and summary. (source: https://docs.databricks.com/aws/en/dev-tools/cli/bundle-commands/)

In [0]:
cli_commands = [
    ("databricks bundle init",          "Scaffold a new bundle from a template",          "Local / one-time"),
    ("databricks bundle validate",      "Check config syntax, variables, references",     "CI: on every pull request"),
    ("databricks bundle deploy -t <t>", "Deploy jobs/pipelines/etc. to a target",         "CD: after merge (dev/test/prod)"),
    ("databricks bundle run <key>",     "Trigger a deployed job or pipeline",             "CD: smoke test after deploy"),
    ("databricks bundle summary",       "Show deployed resources and URLs",               "CI/CD logs"),
    ("databricks bundle destroy -t <t>", "Remove everything the bundle deployed",         "Ephemeral dev/test cleanup"),
    ("databricks jobs list / runs get", "Manage other workspace assets directly",         "Scripts / diagnostics"),
    ("databricks auth login",           "Authenticate (OAuth); CI uses service-principal env vars", "Developer machine / CI"),
]
display(spark.createDataFrame(cli_commands, "command STRING, purpose STRING, where_used_in_ci_cd STRING"))

if shutil.which("databricks"):
    print(subprocess.run(["databricks", "--version"], capture_output=True, text=True, timeout=30).stdout)
else:
    print("CLI not installed here. Install: curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh | sh")

github_actions = '''
name: deploy-bundle
on:
  pull_request:
  push:
    branches: [main]
jobs:
  validate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: databricks/setup-cli@main
      - run: databricks bundle validate -t dev
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.SP_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.SP_CLIENT_SECRET }}
  deploy_prod:
    if: github.ref == 'refs/heads/main'
    needs: validate
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: databricks/setup-cli@main
      - run: databricks bundle deploy -t prod
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.SP_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.SP_CLIENT_SECRET }}
'''
yaml.safe_load(github_actions)           # proves the workflow file is valid YAML
print(github_actions)

---
## Section 6: Troubleshooting, Monitoring, and Optimization (10%)

#### Objective 6.1
> Identify trends in job performance using the Lakeflow Jobs run history view to compare current execution times against historical baselines.

**Job run history** - Every job run is recorded with period start/end times, result state, and durations. The Jobs UI run history view lets you compare current execution times against historical baselines, and the account-wide system.lakeflow.job_run_timeline system table exposes the same data for SQL analysis (365-day retention). (source: https://docs.databricks.com/aws/en/admin/system-tables/jobs/)

In [0]:
import datetime as dt

def get_run_history():
    """Real run history from the system table when available, otherwise a synthetic history with a slow last run."""
    try:
        real = spark.sql("""
            SELECT job_id, run_id,
                   min(period_start_time) AS start_time, max(period_end_time) AS end_time,
                   max_by(result_state, period_end_time) AS result_state
            FROM system.lakeflow.job_run_timeline
            WHERE period_start_time >= current_timestamp() - INTERVAL 30 DAYS
            GROUP BY job_id, run_id
        """)
        if real.limit(1).count() > 0:
            return real, "system.lakeflow.job_run_timeline"
    except Exception as e:
        print("system table not available:", type(e).__name__)
    t0 = dt.datetime(2026, 10, 1, 2, 0)
    rows = []
    for i in range(20):
        duration = 600 + (i % 5) * 20 + (900 if i == 19 else 0)        # last run is much slower
        rows.append((1, i, t0 + dt.timedelta(days=i), t0 + dt.timedelta(days=i, seconds=duration), "SUCCEEDED"))
    return spark.createDataFrame(rows, "job_id LONG, run_id LONG, start_time TIMESTAMP, end_time TIMESTAMP, result_state STRING"), "synthetic demo data"

runs, source = get_run_history()
print("data source:", source)

w = Window.partitionBy("job_id").orderBy("start_time")
trend = (runs
         .withColumn("duration_s", F.col("end_time").cast("long") - F.col("start_time").cast("long"))
         .withColumn("baseline_s", F.avg("duration_s").over(w.rowsBetween(-14, -1)))      # mean of the previous 14 runs
         .withColumn("vs_baseline", F.round(F.col("duration_s") / F.col("baseline_s"), 2))
         .withColumn("regression", F.col("vs_baseline") > 1.5))
display(trend.orderBy(F.desc("start_time")).limit(8))

display(trend.groupBy("job_id").agg(
    F.percentile_approx("duration_s", 0.5).alias("p50_s"),
    F.percentile_approx("duration_s", 0.9).alias("p90_s"),
    F.max("duration_s").alias("max_s"),
    F.sum(F.col("regression").cast("int")).alias("slow_runs")))

#### Objective 6.2
> Use the Lakeflow Jobs UI to monitor pipeline health by interpreting job statuses, viewing DAG-based task graphs to spot upstream blockers, and tracking pipeline run times and failure rates.

**Jobs UI monitoring** - Run details show job statuses (result states) per task and a timeline view to identify tasks that take a long time, understand dependencies and overlap, and spot upstream blockers; for serverless jobs, query statements and profiles are integrated into the timeline view. (source: https://docs.databricks.com/aws/en/jobs/monitor/)

In [0]:
def get_task_runs():
    try:
        real = spark.sql("""
            SELECT job_id, run_id, task_key,
                   max_by(result_state, period_end_time)     AS result_state,
                   max_by(termination_code, period_end_time) AS termination_code,
                   unix_timestamp(max(period_end_time)) - unix_timestamp(min(period_start_time)) AS duration_s
            FROM system.lakeflow.job_task_run_timeline
            WHERE period_start_time >= current_timestamp() - INTERVAL 14 DAYS
            GROUP BY job_id, run_id, task_key
        """)
        if real.limit(1).count() > 0:
            return real, "system.lakeflow.job_task_run_timeline"
    except Exception as e:
        print("system table not available:", type(e).__name__)
    rows = [
        (1, 1, "extract",   "SUCCEEDED", "SUCCESS", 120), (1, 1, "transform", "SUCCEEDED", "SUCCESS", 300), (1, 1, "load", "SUCCEEDED", "SUCCESS", 90),
        (1, 2, "extract",   "SUCCEEDED", "SUCCESS", 130), (1, 2, "transform", "SUCCEEDED", "SUCCESS", 310), (1, 2, "load", "SUCCEEDED", "SUCCESS", 95),
        (1, 3, "extract",   "FAILED",    "RUN_EXECUTION_ERROR", 40),
        (1, 3, "transform", "UPSTREAM_FAILED", "UPSTREAM_FAILED", 0), (1, 3, "load", "UPSTREAM_FAILED", "UPSTREAM_FAILED", 0),
    ]
    return spark.createDataFrame(rows, "job_id LONG, run_id LONG, task_key STRING, result_state STRING, termination_code STRING, duration_s LONG"), "synthetic demo data"

tasks, source = get_task_runs()
print("data source:", source)

print("Status per task:")
display(tasks.groupBy("task_key", "result_state").count().orderBy("task_key", "result_state"))

print("Run time and failure rate per task (Run history view):")
display(tasks.groupBy("task_key").agg(
    F.round(F.avg("duration_s"), 1).alias("avg_run_s"),
    F.round(F.avg((F.col("result_state") == "FAILED").cast("int")) * 100, 1).alias("failure_rate_pct")))

print("Upstream blockers = tasks that failed on their own; their downstream tasks only show UPSTREAM_FAILED:")
display(tasks.filter("result_state = 'FAILED' AND termination_code != 'UPSTREAM_FAILED'")
             .select("job_id", "run_id", F.col("task_key").alias("root_cause_task"), "termination_code"))
display(tasks.filter("termination_code = 'UPSTREAM_FAILED'")
             .select("job_id", "run_id", F.col("task_key").alias("blocked_task")))

#### Objective 6.3
> Identify common performance bottlenecks such as data skew, shuffling, and disk spilling by interpreting stage-level metrics in the Spark UI.

**Data skew** - Uneven key distribution after a shuffle sends most rows to a few partitions, creating straggler tasks while others finish early — visible in the Spark UI as a few tasks reading far more data than the rest. (source: https://spark.apache.org/docs/latest/tuning.html)

**Shuffle and disk spill** - Joins and aggregations force a shuffle (network data exchange between executors); when a task's working set exceeds its memory, Spark spills it to local disk — visible as spill (memory/disk) metrics per task in the stage view, and a major slowdown. (source: https://spark.apache.org/docs/latest/tuning.html)

**Spark UI** - The per-application web UI showing jobs, stages, and tasks with shuffle read/write and spill metrics at task level — the primary tool for finding skew, stragglers, and oversized shuffles at stage level. (source: https://spark.apache.org/docs/latest/web-ui.html)

In [0]:
n = 3_000_000
skewed = (spark.range(n)
          .withColumn("key", F.when(F.col("id") % 10 < 9, F.lit(0).cast("long")).otherwise(F.col("id") % 1000))
          .withColumn("v", F.rand(42)))

print("1) SKEW: one key dominates the data (look for max task duration >> median in the Spark UI stage summary)")
display(skewed.groupBy("key").count().orderBy(F.desc("count")).limit(5))

print("2) Rows per partition after a hash shuffle on the key (the oversized partition = the straggler task)")
display(skewed.repartition(8, "key").withColumn("partition_id", F.spark_partition_id())
              .groupBy("partition_id").count().orderBy("partition_id"))

def timed(label, df):
    t0 = time.time()
    df.write.format("noop").mode("overwrite").save()
    print(f"  {label:<26}{time.time() - t0:6.2f}s")

print("3) SHUFFLE: a wide transformation moves data across the network (Exchange in the plan, Shuffle Read/Write in the UI)")
skewed.groupBy("key").agg(F.sum("v").alias("s")).explain()

print("4) Mitigation by salting: spread the hot key over several partial aggregates, then combine")
plain = skewed.groupBy("key").agg(F.sum("v").alias("s"))
salted = (skewed.withColumn("salt", (F.rand(7) * 8).cast("int"))
                .groupBy("key", "salt").agg(F.sum("v").alias("s"))
                .groupBy("key").agg(F.sum("s").alias("s")))
timed("plain group-by", plain)
timed("salted group-by", salted)
print("  AQE skew-join handling enabled:", conf("spark.sql.adaptive.skewJoin.enabled"))

display(spark.createDataFrame([
    ("Data skew", "Stage > Summary Metrics: max task duration / input size far above median; a few tasks still running", "Salting, AQE skew join, better partition key, broadcast the small side"),
    ("Excessive shuffle", "Stage > Shuffle Read/Write size very large; many Exchange nodes in the SQL plan", "Filter/aggregate earlier, broadcast joins, fewer wide transformations, tune shuffle partitions"),
    ("Disk spilling", "Stage > 'Spill (Memory)' and 'Spill (Disk)' columns > 0", "More partitions, more memory per core, avoid huge groups / collect_list, bigger node type"),
    ("Small files / too many tasks", "Thousands of tiny tasks, high scheduler delay", "OPTIMIZE / Liquid Clustering, coalesce, AQE partition coalescing"),
], "bottleneck STRING, where_to_see_it_in_spark_ui STRING, typical_fix STRING"))

#### Objective 6.4
> Understand the features of Liquid Clustering and predictive optimization.

**Liquid clustering** - Delta Lake's data layout technique that co-locates related data on the keys you choose with CLUSTER BY, replacing partitioning and ZORDER; data is incrementally re-clustered as it arrives, improving selective query performance without over-partitioning. (source: https://docs.databricks.com/aws/en/tables/clustering/)

**Predictive optimization** - Automatically runs table maintenance (OPTIMIZE, VACUUM) in the background on Unity Catalog managed tables, analyzing usage history to keep tables well-clustered and compact without manual scheduling. (source: https://docs.databricks.com/aws/en/optimizations/predictive-optimization/)

In [0]:
spark.sql("DROP TABLE IF EXISTS lc_events")
# Liquid Clustering replaces partitioning + ZORDER: flexible clustering keys declared with CLUSTER BY
spark.sql("""CREATE TABLE lc_events (event_id BIGINT, user_id INT, event_date DATE, payload STRING)
             CLUSTER BY (user_id, event_date)""")
(spark.range(1_000_000)
      .select(F.col("id").alias("event_id"),
              (F.col("id") % 5000).cast("int").alias("user_id"),
              F.expr("date_add(DATE'2026-01-01', cast(id % 300 AS INT))").alias("event_date"),
              F.lit("x").alias("payload"))
      .write.mode("append").saveAsTable("lc_events"))

display(spark.sql("OPTIMIZE lc_events"))                                  # incremental clustering of new data
display(spark.sql("DESCRIBE DETAIL lc_events").select("clusteringColumns", "numFiles", "sizeInBytes"))

# Keys can change later without rewriting the whole table (existing data is re-clustered gradually)
spark.sql("ALTER TABLE lc_events CLUSTER BY (event_date)")
display(spark.sql("DESCRIBE DETAIL lc_events").select("clusteringColumns"))

# Let Databricks choose / adapt the clustering keys from query patterns (needs predictive optimization)
try_run("CLUSTER BY AUTO", lambda: spark.sql("ALTER TABLE lc_events CLUSTER BY AUTO"))

# Predictive optimization = Databricks automatically runs OPTIMIZE, VACUUM (and stats) on UC managed tables
try_run("schema PO status", lambda: display(
    spark.sql(f"DESCRIBE SCHEMA EXTENDED `{CATALOG}`.`{SCHEMA}`").filter("info_name LIKE '%redictive%'")))
try_run("PO history", lambda: display(spark.sql(f"""
    SELECT start_time, table_name, operation_type, operation_status
    FROM system.storage.predictive_optimization_operations_history
    WHERE catalog_name = '{CATALOG}' AND schema_name = '{SCHEMA}'
    ORDER BY start_time DESC LIMIT 10""")))
if RUN_SIDE_EFFECTS:
    try_run("enable PO", lambda: spark.sql(f"ALTER SCHEMA `{CATALOG}`.`{SCHEMA}` ENABLE PREDICTIVE OPTIMIZATION"))
else:
    print("To enable: ALTER SCHEMA <catalog>.<schema> ENABLE PREDICTIVE OPTIMIZATION  (set run_side_effects=true to run it here)")

#### Objective 6.5
> Diagnose cluster startup failures, library conflicts, and out-of-memory issues.

**Cluster startup failures** - Common causes include cloud/network timeouts, too many libraries or blocking init scripts at launch, and cloud-provider instance limits or shutdowns; typically fixed by reducing libraries, using cluster-scoped init scripts, or retrying. (source: https://kb.databricks.com/clusters/cluster-failed-launch)

**Library conflicts** - Version clashes when installed packages or their transitive dependencies are incompatible; resolved by pinning versions, excluding conflicting dependencies, or using cluster-scoped libraries. (source: https://kb.databricks.com/libraries/failed-to-install-elasticsearch-via-maven)

**Out-of-memory (OOM)** - Driver or executor JVM heap exhaustion; on the driver often caused by collecting large results (collect/toPandas) or wide operations, and skewed data can also cause OOM errors — fix by reducing collection size, increasing memory/partitions, or fixing the skew. (source: https://spark.apache.org/docs/latest/tuning.html)

In [0]:
import importlib.metadata as md, itertools

# 1) Cluster startup failures -> cluster event log (classic compute only)
cid = conf("spark.databricks.clusterUsageTags.clusterId")
if cid:
    w = get_workspace_client()
    events = try_run("cluster events", lambda: list(itertools.islice(w.clusters.events(cluster_id=cid), 15))) or []
    for e in events:
        print(e.timestamp, getattr(e.type, "value", e.type), str(e.details)[:160])
else:
    print("Serverless compute has no cluster event log; check job run output / query history instead.")

# 2) Library conflicts -> compare installed versions and run pip's dependency checker
for pkg in ["pandas", "numpy", "pyarrow", "pyspark"]:
    try:
        print(f"{pkg:<10}{md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:<10}not installed")
check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True, timeout=120)
print("pip check ->", (check.stdout.strip() or "no broken requirements")[:1500])

# 3) Out-of-memory -> know the driver limits and never pull big data to the driver
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"driver memory: {vm.total / 2**30:.1f} GiB total, {vm.available / 2**30:.1f} GiB available")
except Exception as e:
    print("psutil unavailable:", type(e).__name__)
print("spark.driver.maxResultSize =", conf("spark.driver.maxResultSize"))

big = spark.range(50_000_000)
# BAD  : big.collect() / big.toPandas()  -> driver OOM (or maxResultSize exceeded)
sample = big.limit(1000).toPandas()                    # GOOD: bound what reaches the driver
print("rows safely brought to the driver:", len(sample))

display(spark.createDataFrame([
    ("Cluster won't start / terminates at startup", "Cluster event log: reason + code (quota, subnet/IP exhaustion, bad init script, invalid config)", "Fix the cause from the event; test init scripts; check cloud quotas and instance availability"),
    ("ImportError / version conflict after installing a library", "Driver log, `pip check`, library status in the cluster UI", "Pin versions, use cluster-scoped libraries or a serverless environment, avoid mixing %pip with clashing wheels"),
    ("Driver OOM", "Driver logs / metrics tab; failure on collect(), toPandas(), broadcast of a large table", "Aggregate or limit before collecting, lower autoBroadcastJoinThreshold, bigger driver"),
    ("Executor OOM / container killed", "Stage shows failed tasks 'ExecutorLostFailure', large spill", "More partitions, fix skew, larger or memory-optimized nodes, avoid huge groups"),
], "symptom STRING, where_to_look STRING, typical_remedy STRING"))

---
## Section 7: Governance and Security (15%)

#### Objective 7.1
> Differentiate between managed and external tables in Unity Catalog and perform basic operations (create, modify, delete, and convert between managed and external tables) on them.

**Managed table** - A table where Unity Catalog determines the storage location: data lives in the managed storage of the containing schema/catalog, uses Delta or Iceberg format, and dropping the table deletes the underlying files (after a recovery period of 7 days by default). (source: https://docs.databricks.com/aws/en/data-governance/unity-catalog/managed-versus-external/)

**External table** - A table whose storage location you specify (a path covered by a UC external location); Unity Catalog governs access but not the data's lifecycle — dropping the table removes only the metadata, the files remain. (source: https://docs.databricks.com/aws/en/data-governance/unity-catalog/managed-versus-external/)

In [0]:
# MANAGED table: Unity Catalog owns the storage location and the data lifecycle
spark.sql("CREATE OR REPLACE TABLE uc_managed AS SELECT id, id * 2 AS v FROM range(5)")
spark.sql("INSERT INTO uc_managed VALUES (100, 200)")                          # modify
spark.sql("UPDATE uc_managed SET v = -1 WHERE id = 0")
display(spark.sql("DESCRIBE EXTENDED uc_managed").filter("col_name IN ('Type', 'Provider', 'Location')"))

# EXTERNAL table: you own the storage path (needs an external location + CREATE EXTERNAL TABLE privilege)
w = get_workspace_client()
locations = try_run("external locations", lambda: list(w.external_locations.list())) or []
ext_path = None
if locations:
    ext_path = f"{locations[0].url.rstrip('/')}/{SCHEMA}/uc_external"
    spark.sql("DROP TABLE IF EXISTS uc_external")
    try_run("create external table", lambda: (spark.range(5).withColumn("v", F.col("id") * 3)
            .write.format("delta").mode("overwrite").option("path", ext_path).saveAsTable("uc_external")))
    ext_ok = spark.catalog.tableExists("uc_external")   # creation can fail, e.g. no CREATE EXTERNAL TABLE privilege
    if ext_ok:
        display(spark.sql("DESCRIBE EXTENDED uc_external").filter("col_name IN ('Type', 'Provider', 'Location')"))
    else:
        print("External table could not be created (missing CREATE EXTERNAL TABLE privilege on the location).")

    # Managed -> external: copy into a table that has a LOCATION
    ctas_path = f"{locations[0].url.rstrip('/')}/{SCHEMA}/uc_managed_as_external"
    try_run("managed -> external", lambda: spark.sql(
        f"CREATE OR REPLACE TABLE uc_was_managed LOCATION '{ctas_path}' AS SELECT * FROM uc_managed"))
    # External -> managed
    try_run("external -> managed", lambda: spark.sql("ALTER TABLE uc_external SET MANAGED"))
    if ext_ok:
        display(spark.sql("DESCRIBE EXTENDED uc_external").filter("col_name = 'Type'"))
else:
    print("No external location available to you -> showing the SQL only:")
    print("  CREATE TABLE ext_t (id INT) LOCATION 'abfss://c@acct.dfs.core.windows.net/path'   -- external")
    print("  ALTER TABLE ext_t SET MANAGED                                                      -- external -> managed")
    print("  CREATE TABLE ext_copy LOCATION '<path>' AS SELECT * FROM managed_t                 -- managed -> external (copy)")

# DELETE semantics
spark.sql("DROP TABLE uc_managed")                      # managed: data is removed after a retention period
try_run("show dropped", lambda: display(spark.sql(f"SHOW TABLES DROPPED IN `{CATALOG}`.`{SCHEMA}`")))
try_run("undrop", lambda: spark.sql("UNDROP TABLE uc_managed"))                 # managed drops are recoverable for ~7 days
print("external DROP removes only the metadata - the files at LOCATION stay in your storage.")

#### Objective 7.2
> Configure access controls using the UI and SQL by applying GRANT, REVOKE, and DENY privileges to principals (users, groups, and service principals) at appropriate levels of the security hierarchy.

**GRANT / REVOKE** - SQL statements that give and remove privileges (SELECT, MODIFY, CREATE TABLE, USE CATALOG, USE SCHEMA, ...) on securable objects for principals — users, groups, and service principals — at any level of the hierarchy: metastore > catalog > schema > table. (source: https://docs.databricks.com/aws/en/data-governance/unity-catalog/manage-privileges/)

**DENY** - An explicit privilege denial that blocks access even where a GRANT would allow it; available as SQL DENY statements on securables and as ABAC DENY policies (Beta). (source: https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/)

In [0]:
me = spark.sql("SELECT current_user()").first()[0]
dbutils.widgets.text("principal", "", "Principal to grant to (blank = you)")
principal = dbutils.widgets.get("principal") or me
P = f"`{principal}`"          # users: email; groups: group name; service principals: application ID

spark.sql("CREATE OR REPLACE TABLE sec_demo AS SELECT 1 AS id, 'x' AS v")

display(spark.createDataFrame([
    ("Metastore", "CREATE CATALOG, CREATE EXTERNAL LOCATION ...", "inherited by every catalog below"),
    ("Catalog", "USE CATALOG, CREATE SCHEMA", "needed to reach anything inside it"),
    ("Schema", "USE SCHEMA, CREATE TABLE, SELECT (on all tables)", "inherited by tables/views/volumes in the schema"),
    ("Table / view", "SELECT, MODIFY", "most specific level"),
], "securable STRING, example_privileges STRING, note STRING"))
print("Privileges are INHERITED downward; to read a table you need USE CATALOG + USE SCHEMA + SELECT.\n")

statements = [
    f"GRANT USE CATALOG ON CATALOG `{CATALOG}` TO {P}",
    f"GRANT USE SCHEMA ON SCHEMA `{CATALOG}`.`{SCHEMA}` TO {P}",
    f"GRANT SELECT ON TABLE sec_demo TO {P}",
    f"GRANT MODIFY ON TABLE sec_demo TO {P}",
    f"REVOKE MODIFY ON TABLE sec_demo FROM {P}",
    f"DENY SELECT ON TABLE sec_demo TO {P}",
]
for s in statements:
    print("SQL>", s)
    try_run("statement", lambda s=s: spark.sql(s))

try_run("show grants", lambda: display(spark.sql("SHOW GRANTS ON TABLE sec_demo")))
print("\nNote: DENY exists for legacy table ACLs; Unity Catalog manages access with GRANT/REVOKE (plus ABAC policies, see 7.4).")
print("If the DENY above was rejected, that is the expected result in Unity Catalog.")
print("Groups: GRANT ... TO `data_analysts` | Service principals: GRANT ... TO `<application-id>`")

#### Objective 7.3
> Understand column-level masking and row-level security to restrict data visibility based on user groups.

**Row filter** - A Unity Catalog function applied to a table that restricts which rows each user sees at query time (e.g., rows only for the user's group), evaluated with the invoker's identity (current_user, is_member). (source: https://docs.databricks.com/aws/en/data-governance/unity-catalog/filters-and-masks/)

**Column mask** - A Unity Catalog function applied to a column that masks its values at query time, showing full values only to authorized users/groups and masked values to everyone else. (source: https://docs.databricks.com/aws/en/data-governance/unity-catalog/filters-and-masks/)

In [0]:
spark.sql("CREATE OR REPLACE TABLE customers_pii (id INT, name STRING, ssn STRING, region STRING)")
spark.sql("""INSERT INTO customers_pii VALUES
             (1, 'Ada',   '111-22-3333', 'EU'),
             (2, 'Grace', '444-55-6666', 'US'),
             (3, 'Linus', '777-88-9999', 'EU')""")
print("Before any protection:"); display(spark.table("customers_pii"))

# Column mask: function decides what each viewer sees, based on group membership
spark.sql("""CREATE OR REPLACE FUNCTION mask_ssn(ssn STRING) RETURNS STRING
             RETURN CASE WHEN is_account_group_member('pii_admins') THEN ssn ELSE concat('***-**-', right(ssn, 4)) END""")
# Row filter: function returns TRUE for rows the viewer may see
spark.sql("""CREATE OR REPLACE FUNCTION region_filter(region STRING) RETURNS BOOLEAN
             RETURN is_account_group_member('global_analysts') OR region = 'EU'""")

spark.sql("ALTER TABLE customers_pii ALTER COLUMN ssn SET MASK mask_ssn")
spark.sql("ALTER TABLE customers_pii SET ROW FILTER region_filter ON (region)")
print("After (you are not in pii_admins / global_analysts -> masked SSNs and EU rows only):")
display(spark.table("customers_pii"))

# Remove the protections again
spark.sql("ALTER TABLE customers_pii ALTER COLUMN ssn DROP MASK")
spark.sql("ALTER TABLE customers_pii DROP ROW FILTER")
print("Protections removed:"); display(spark.table("customers_pii"))

#### Objective 7.4
> Understand Unity Catalog ABAC policies to centrally control row-level filtering and column masking for sensitive data.

**ABAC (attribute-based access control)** - An access-control model in Unity Catalog where access is determined by policies evaluated against attributes — represented through governed tags — on securable objects. A single row-filter or column-mask policy attached at a catalog or schema level applies automatically to every matching table; ABAC also supports GRANT policies for dynamic privilege grants and DENY policies (Beta) for explicit denials. (source: https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/)

In [0]:
# ABAC = attribute-based: tag data once, write ONE policy at catalog/schema level that applies to every matching table
spark.sql("CREATE OR REPLACE TABLE abac_demo (id INT, ssn STRING, region STRING)")
spark.sql("INSERT INTO abac_demo VALUES (1, '111-22-3333', 'EU'), (2, '444-55-6666', 'US')")
spark.sql("""CREATE OR REPLACE FUNCTION abac_mask_ssn(ssn STRING) RETURNS STRING
             RETURN concat('***-**-', right(ssn, 4))""")
spark.sql("""CREATE OR REPLACE FUNCTION abac_region_filter(region STRING) RETURNS BOOLEAN
             RETURN region = 'EU'""")

# Step 1: classify data with tags (governed tags are the recommended basis for ABAC)
try_run("tag column", lambda: spark.sql("ALTER TABLE abac_demo ALTER COLUMN ssn SET TAGS ('pii' = 'ssn')"))
try_run("tag column", lambda: spark.sql("ALTER TABLE abac_demo ALTER COLUMN region SET TAGS ('geo' = 'region')"))

# Step 2: policies. Syntax follows the Unity Catalog ABAC docs; features/syntax can differ by release - check the docs.
mask_policy = f"""
CREATE OR REPLACE POLICY mask_pii_columns
ON SCHEMA `{CATALOG}`.`{SCHEMA}`
COMMENT 'Mask columns tagged pii for everyone except pii_admins'
COLUMN MASK `{CATALOG}`.`{SCHEMA}`.abac_mask_ssn
TO `account users`
EXCEPT `pii_admins`
FOR TABLES
MATCH COLUMNS hasTag('pii') AS pii_col
ON COLUMN pii_col
"""
row_policy = f"""
CREATE OR REPLACE POLICY eu_rows_only
ON SCHEMA `{CATALOG}`.`{SCHEMA}`
COMMENT 'Non-global analysts only see EU rows'
ROW FILTER `{CATALOG}`.`{SCHEMA}`.abac_region_filter
TO `account users`
EXCEPT `global_analysts`
FOR TABLES
MATCH COLUMNS hasTag('geo') AS geo_col
USING COLUMNS (geo_col)
"""
for sql_text in (mask_policy, row_policy):
    print(sql_text)
    if RUN_SIDE_EFFECTS:
        try_run("create policy", lambda s=sql_text: spark.sql(s))
    else:
        print("-> printed only (set run_side_effects=true to create the policy in your sandbox schema)\n")

try_run("show tags", lambda: display(spark.sql(
    f"SELECT column_name, tag_name, tag_value FROM `{CATALOG}`.information_schema.column_tags "
    f"WHERE schema_name = '{SCHEMA}' AND table_name = 'abac_demo'")))

display(spark.createDataFrame([
    ("Where defined", "On each table (ALTER TABLE ... SET MASK / ROW FILTER)", "Once at catalog/schema level as a policy"),
    ("Scales to new tables", "No - repeat per table", "Yes - any table with matching tags is covered automatically"),
    ("Driven by", "Explicit table + column", "Tags / attributes of the data"),
    ("Best for", "A few special tables", "Org-wide sensitive-data rules (PII, regions)"),
], "aspect STRING, table_level_mask_and_filter STRING, abac_policy STRING"))

---
## Cleanup (optional)

Not part of the exam outline — removes the sandbox objects created by this notebook.

In [0]:
# Uncomment to delete everything this notebook created (tables, views, functions, volume files, policies).
# for p in ("mask_pii_columns", "eu_rows_only"):
#     try_run("drop policy", lambda p=p: spark.sql(f"DROP POLICY IF EXISTS {p} ON SCHEMA `{CATALOG}`.`{SCHEMA}`"))

# spark.sql(f"DROP SCHEMA IF EXISTS `{CATALOG}`.`{SCHEMA}` CASCADE")
shutil.rmtree("/tmp/de_associate_bundle", ignore_errors=True)
print("Sandbox schema kept:", f"{CATALOG}.{SCHEMA}")